# Tier 1 + retr. — passage-order sensitivity, Ollama model (Llama-3.1-8B)

Re-runs the Tier 1 retrieval-augmented items with the retrieved passages in **random** and **reverse** order (preprint §4.4.3,
Table 4). The code is copied verbatim from `run_retrieved_chunk_order_tier1.py` and `retrieved_chunk_order.py`; its
`main()` is replaced by a run cell that reads from and writes to `data-generated/`, and writes each order to its own folder.
One line is changed for Ollama: the token limit is passed as `max_tokens` (as in the Tier 1 Ollama scripts) instead of
`max_completion_tokens`.

| Output folder | Passage order |
|---|---|
| `data-generated/tier1_retr_chunk_random_order/` | seeded shuffle per vignette (`random_seed + vignette_idx`, shared across the 8 demographic variants) |
| `data-generated/tier1_retr_chunk_reverse_order/` | retrieved passages reversed |

Source (template) files: `data-generated/tier1_orig/results_post_op_<model>_retr_evidence_tier1_orig_ff.csv`, i.e. the
"+ retr." Tier 1 outputs written by the Tier 1 notebook. Each row reuses that row's closed prompt and its 10 retrieved
passages; only their order in the open prompt changes. The original order is **not** re-run: compare against the source file.

Output files: `results_post_op_<model>_retr_evidence_tier1_chunk_<random|reverse>_order_ff.csv` (80 rows each).

1 model × 2 orders × 80 = **160 calls**. Re-running resumes: rows already in an output file are skipped.

Settings follow the chunk-order script, not the Tier 1 notebook: 1200 tokens, no log probabilities
(the `prob_*` columns are copied from the source row, not recomputed), and replies are cut at `##`.

**Requirements:** Ollama running locally (`ollama serve`) with `llama3.1:8b` pulled; the Tier 1 notebook's `+ retr.` `tier1_orig` outputs.

## 1. Setup

In [ ]:
from __future__ import annotations

import glob
import os
import sys
import time
from typing import Any

import pandas as pd
from openai import OpenAI


In [ ]:
# ==== Configuration ====
# Ollama model id -> tag used in the Tier 1 file names
MODELS = {
    "llama3.1:8b": "llama3_1",
}
PERMUTATIONS_TO_RUN = ["random", "reverse"]

# Script defaults (argparse defaults in run_retrieved_chunk_order_tier1.py)
RANDOM_SEED = 42               # --random-seed
TEMPERATURE = 0.0              # --temperature
MAX_COMPLETION_TOKENS = 1200   # --max-completion-tokens
SLEEP_S = 1.0                  # --sleep
DRY_RUN = False                # --dry-run: build prompts only, no API calls, nothing written
LIMIT = None                   # --limit: max rows per model (testing)
RESUME = True                  # not --no-resume: skip rows already in the output CSV

# ==== Paths ====
PROJECT_ROOT = os.getcwd() if os.path.isdir(os.path.join(os.getcwd(), "data-generated")) else os.path.dirname(os.getcwd())
OUTPUT_ROOT = os.path.join(PROJECT_ROOT, "data-generated")
SOURCE_DIR = os.path.join(OUTPUT_ROOT, "tier1_orig")
OUT_DIR_BY_PERM = {
    "random":  os.path.join(OUTPUT_ROOT, "tier1_retr_chunk_random_order"),
    "reverse": os.path.join(OUTPUT_ROOT, "tier1_retr_chunk_reverse_order"),
}

def source_csv_for(tag):
    return os.path.join(SOURCE_DIR, f"results_post_op_{tag}_retr_evidence_tier1_orig_ff.csv")

def out_csv_for(tag, perm):
    return os.path.join(OUT_DIR_BY_PERM[perm], f"results_post_op_{tag}_retr_evidence_tier1_chunk_{perm}_order_ff.csv")

print("Output root:", OUTPUT_ROOT)
for model_id, tag in MODELS.items():
    p = source_csv_for(tag)
    print(("✓ " if os.path.exists(p) else "✗ MISSING ") + os.path.relpath(p, PROJECT_ROOT))


In [ ]:
# Connect to local Ollama instance via its OpenAI-compatible API
client: OpenAI | None = None
if not DRY_RUN:
    client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")


## 2. Verbatim code

In [ ]:
# ==== Verbatim: retrieved_chunk_order.py (everything after its docstring) ====
from __future__ import annotations

import random
import re
from dataclasses import dataclass
from typing import Literal

PermutationMode = Literal["reverse", "random"]



@dataclass(frozen=True)
class RetrievedPromptParts:
    """Split of a retrieved open prompt."""

    prefix: str
    chunks: tuple[tuple[str, str], ...]  # (tag, body) — last body excludes </context>
    tail: str
    context_close_suffix: str  # e.g. "</context>\\n\\n" between chunks and vignette

    @property
    def chunk_ids(self) -> tuple[str, ...]:
        return tuple(tag for tag, _ in self.chunks)

    def chunk_order_str(self) -> str:
        return ";".join(self.chunk_ids)


def parse_retrieved_open_prompt(open_prompt: str) -> RetrievedPromptParts:
    """
  Split open_prompt into:
    - prefix: through opening <context> header (before first [pdf_...] tag)
    - chunks: list of (tag, body) in current order
    - tail: from first 'Vignette:' onward
    """
    if not open_prompt or not isinstance(open_prompt, str):
        raise ValueError("open_prompt must be a non-empty string")

    vpos = open_prompt.find("Vignette:")
    if vpos < 0:
        raise ValueError("Could not find 'Vignette:' in open_prompt")

    ctx = open_prompt[:vpos]
    tail = open_prompt[vpos:]

    parts = re.split(r"(\[pdf_[^\]]+\])", ctx)
    if len(parts) < 3:
        raise ValueError(f"Expected at least one [pdf_...] chunk tag, got {len(parts) - 1} tags")

    prefix = parts[0]
    chunks: list[tuple[str, str]] = []

    for i in range(1, len(parts), 2):
        if i + 1 >= len(parts):
            break
        chunks.append((parts[i], parts[i + 1]))

    if not chunks:
        raise ValueError("No retrieved chunks parsed from open_prompt")

    context_close_suffix = ""
    last_tag, last_body = chunks[-1]
    close_match = re.search(r"(</context>\s*)$", last_body, flags=re.IGNORECASE | re.DOTALL)
    if close_match:
        context_close_suffix = close_match.group(1)
        last_body = last_body[: close_match.start()]
        chunks[-1] = (last_tag, last_body)

    return RetrievedPromptParts(
        prefix=prefix,
        chunks=tuple(chunks),
        tail=tail,
        context_close_suffix=context_close_suffix,
    )


def permute_chunks(
    chunks: tuple[tuple[str, str], ...],
    mode: PermutationMode,
    *,
    vignette_idx: int,
    random_seed: int = 42,
) -> tuple[tuple[str, str], ...]:
    """Return chunks in permuted order (same multiset of tag/body pairs)."""
    items = list(chunks)
    if mode == "reverse":
        return tuple(reversed(items))
    if mode == "random":
        rng = random.Random(random_seed + int(vignette_idx))
        order = list(range(len(items)))
        rng.shuffle(order)
        return tuple(items[i] for i in order)
    raise ValueError(f"Unknown permutation mode: {mode}")


def reassemble_open_prompt(parts: RetrievedPromptParts, chunks: tuple[tuple[str, str], ...]) -> str:
    """Rebuild open_prompt from prefix, reordered chunks, and vignette tail."""
    if not chunks:
        raise ValueError("chunks must be non-empty")

    out = parts.prefix
    for tag, body in chunks:
        out += tag + body
    out += parts.context_close_suffix
    return out + parts.tail.lstrip()


def apply_chunk_permutation(
    open_prompt: str,
    mode: PermutationMode,
    *,
    vignette_idx: int,
    random_seed: int = 42,
) -> tuple[str, str, str]:
    """
    Permute chunk order in open_prompt.

    Returns:
        new_open_prompt, original_chunk_order (semicolon-separated),
        permuted_chunk_order (semicolon-separated)
    """
    parts = parse_retrieved_open_prompt(open_prompt)
    original_order = parts.chunk_order_str()
    new_chunks = permute_chunks(
        parts.chunks,
        mode,
        vignette_idx=vignette_idx,
        random_seed=random_seed,
    )
    new_open = reassemble_open_prompt(parts, new_chunks)
    new_order = ";".join(tag for tag, _ in new_chunks)
    return new_open, original_order, new_order


In [ ]:
# ==== Verbatim: run_retrieved_chunk_order_tier1.py (helpers + run_model; main() is replaced below) ====
PERMUTATIONS = ("reverse", "random")


def _detect_output_prefix(df: pd.DataFrame) -> str:
    for c in df.columns:
        if str(c).lower().endswith("_answer"):
            return str(c)[: -len("_answer")]
    return "gpt4o"


def _parse_model_output(gen_text: str) -> tuple[str, str, str]:
    if not gen_text:
        return "", "", ""
    lines = gen_text.strip().split("\n")
    answer, dosage, explanation = "", "", ""
    for line in lines:
        if line.startswith("Answer:"):
            answer = line.replace("Answer:", "", 1).strip()
        elif line.startswith("Dosage:"):
            dosage = line.replace("Dosage:", "", 1).strip()
        elif line.startswith("Explanation:"):
            explanation = line.replace("Explanation:", "", 1).strip()
    if not answer and lines:
        answer = lines[0].strip()
    if not dosage and len(lines) > 1:
        dosage = lines[1].strip()
    return answer, dosage, explanation


def _call_openai(
    client: OpenAI,
    *,
    model_id: str,
    final_prompt: str,
    temperature: float,
    max_completion_tokens: int,
    max_retries: int = 5,
) -> str:
    last_err: Exception | None = None
    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model=model_id,
                messages=[{"role": "user", "content": final_prompt}],
                temperature=temperature,
                max_tokens=max_completion_tokens,  # CHANGED for Ollama: was max_completion_tokens=
            )
            text = (resp.choices[0].message.content or "").strip()
            if "##" in text:
                text = text.split("##", 1)[0].strip()
            return text
        except Exception as e:
            last_err = e
            if attempt < max_retries - 1:
                sleep_time = 2 * (2**attempt)
                print(f"    API error (attempt {attempt + 1}/{max_retries}): {e} — sleeping {sleep_time}s")
                time.sleep(sleep_time)
            else:
                raise
    raise RuntimeError(str(last_err))


def _row_key(row: pd.Series) -> tuple:
    return (
        int(row["vignette_idx"]),
        str(row["race"]),
        str(row["gender"]),
        str(row.get("name", "")),
    )


def _load_done_keys(out_path: str) -> set[tuple]:
    if not os.path.isfile(out_path):
        return set()
    done = pd.read_csv(out_path)
    if "permutation_id" not in done.columns:
        return set()
    keys: set[tuple] = set()
    for _, r in done.iterrows():
        keys.add((str(r["permutation_id"]), *_row_key(r)))
    return keys


def run_model(
    *,
    model_folder: str,
    source_csv: str,
    out_path: str,
    client: OpenAI | None,
    model_id: str,
    permutations: tuple[str, ...],
    random_seed: int,
    temperature: float,
    max_completion_tokens: int,
    sleep_s: float,
    dry_run: bool,
    limit: int | None,
    resume: bool,
) -> int:
    df = pd.read_csv(source_csv)
    required = {"vignette_idx", "race", "gender", "closed_prompts", "open_prompts"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{model_folder}: missing columns {sorted(missing)}")

    prefix = _detect_output_prefix(df)
    ans_col, dos_col, expl_col, full_col = (
        f"{prefix}_answer",
        f"{prefix}_dosage",
        f"{prefix}_explanation",
        f"{prefix}_full",
    )

    done_keys: set[tuple] = _load_done_keys(out_path) if resume else set()
    n_calls = 0
    rows_out: list[dict[str, Any]] = []

    work = df.head(limit) if limit is not None else df

    for perm in permutations:
        print(f"  Permutation: {perm}")
        for idx, row in work.iterrows():
            key = (perm, *_row_key(row))
            if key in done_keys:
                continue

            v_idx = int(row["vignette_idx"])
            try:
                new_open, orig_order, perm_order = apply_chunk_permutation(
                    str(row["open_prompts"]),
                    perm,  # type: ignore[arg-type]
                    vignette_idx=v_idx,
                    random_seed=random_seed,
                )
            except Exception as e:
                print(f"    SKIP row {idx} (v{v_idx} {row['race']} {row['gender']}): parse error {e}")
                continue

            final_prompt = str(row["closed_prompts"]) + new_open
            gen_text = ""
            if not dry_run:
                if client is None:
                    raise RuntimeError("OpenAI client required when not in --dry-run mode")
                gen_text = _call_openai(
                    client,
                    model_id=model_id,
                    final_prompt=final_prompt,
                    temperature=temperature,
                    max_completion_tokens=max_completion_tokens,
                )
                if sleep_s > 0:
                    time.sleep(sleep_s)

            answer, dosage, explanation = _parse_model_output(gen_text)

            out_row: dict[str, Any] = {
                "context": row.get("context", "Postoperative Pain"),
                "vignette_idx": v_idx,
                "name": row.get("name", ""),
                "gender": row["gender"],
                "race": row["race"],
                "permutation_id": perm,
                "baseline_chunk_order": orig_order,
                "chunk_order": perm_order,
                "source_model_folder": model_folder,
                "source_tier1_csv": os.path.basename(source_csv),
                ans_col: answer,
                dos_col: dosage,
                expl_col: explanation,
                full_col: gen_text,
                "closed_prompts": row["closed_prompts"],
                "open_prompts": new_open,
            }
            # Preserve prob_* columns if present (not re-computed)
            for c in df.columns:
                if c.startswith("prob_") and c not in out_row:
                    out_row[c] = row[c]

            rows_out.append(out_row)
            n_calls += 1

            if n_calls % 10 == 0:
                print(f"    completed {n_calls} new calls ({perm})")

    if rows_out and not dry_run:
        os.makedirs(os.path.dirname(out_path), exist_ok=True)
        new_df = pd.DataFrame(rows_out)
        if os.path.isfile(out_path):
            existing = pd.read_csv(out_path)
            new_df = pd.concat([existing, new_df], ignore_index=True)
        new_df.to_csv(out_path, index=False)

    return n_calls


## 3. Run all models × passage orders

In [ ]:
# ---- replaces main(): one output file per model × permutation ----
total_calls = 0
print(f"Permutations: {PERMUTATIONS_TO_RUN}")
print(f"Random seed base: {RANDOM_SEED}")
if DRY_RUN:
    print("DRY RUN — no API calls")

for model_id, tag in MODELS.items():
    source_csv = source_csv_for(tag)
    if not os.path.isfile(source_csv):
        print(f"SKIP {model_id}: no Tier 1 + retr. CSV ({source_csv})")
        continue
    n_source = len(pd.read_csv(source_csv))

    for perm in PERMUTATIONS_TO_RUN:
        out_path = out_csv_for(tag, perm)
        print(f"\n{'=' * 72}")
        print(f"MODEL: {model_id}  |  PERMUTATION: {perm}")
        print(f"  Source: {os.path.relpath(source_csv, PROJECT_ROOT)} ({n_source} rows)")
        print(f"  Output: {os.path.relpath(out_path, PROJECT_ROOT)}")
        print(f"  New API calls (max): {n_source}")

        n = run_model(
            model_folder=os.path.basename(SOURCE_DIR),
            source_csv=source_csv,
            out_path=out_path,
            client=client,
            model_id=model_id,
            permutations=(perm,),
            random_seed=RANDOM_SEED,
            temperature=TEMPERATURE,
            max_completion_tokens=MAX_COMPLETION_TOKENS,
            sleep_s=SLEEP_S,
            dry_run=DRY_RUN,
            limit=LIMIT,
            resume=RESUME,
        )
        total_calls += n
        print(f"  Done: {n} new calls this run")

print(f"\nTotal new API calls this run: {total_calls}")


## 4. Sanity check

In [ ]:
# ==== Check: % Low and decision shift vs. the original order (cf. Table 4) ====
# Decision per row: "Denied" if the answer starts with No, otherwise Low/High from the dosage (then answer/full reply).
# Note: GPT-5.4 outputs are only fully comparable after running the post-processing notebook on these folders too.
import re

def decision(row):
    ans = str(row.get("gpt4o_answer", "")).strip().lower()
    if re.match(r"^\W*no\b", ans):
        return "Denied"
    for col in ["gpt4o_dosage", "gpt4o_answer", "gpt4o_full"]:
        s = str(row.get(col, "")).lower()
        if re.search(r"\blow\b", s):
            return "Low"
        if re.search(r"\bhigh\b", s):
            return "High"
    return "Unparsed"

keys = ["vignette_idx", "race", "gender", "name"]
rows = []
for model_id, tag in MODELS.items():
    if not os.path.isfile(source_csv_for(tag)):
        continue
    base = pd.read_csv(source_csv_for(tag))
    base["decision"] = base.apply(decision, axis=1)
    for perm in PERMUTATIONS_TO_RUN:
        p = out_csv_for(tag, perm)
        if not os.path.isfile(p):
            continue
        new = pd.read_csv(p)
        new["decision"] = new.apply(decision, axis=1)
        m = base[keys + ["decision"]].merge(new[keys + ["decision"]], on=keys, suffixes=("_orig", "_reord"))
        rows.append({
            "model": model_id, "order": perm, "n": len(m),
            "% Low original": round(100 * (m.decision_orig == "Low").mean(), 1),
            "% Low reordered": round(100 * (m.decision_reord == "Low").mean(), 1),
            "Shift (%)": round(100 * (m.decision_orig != m.decision_reord).mean(), 1),
            "% unparsed (reordered)": round(100 * (m.decision_reord == "Unparsed").mean(), 1),
        })
pd.DataFrame(rows)
